In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.options.display.max_columns = None
pd.options.display.max_rows = None


### READ DATA

Read cleaned data from data/02_intermediate folder

In [14]:
matches_features = pd.read_csv("/Users/martapriv/Documents/PUM/inzynierka_code/Tennis_Scores_Predictions/data/02_intermediate/matches_cleaned.csv", index_col=0)

### TRANSFORM COLUMN NAMES

Instead of winner_name and loser_name creates new features: player_A, player_B and player_A_won.  
Player names are stored in the following columns: ['winner_name'] and ['loser_name'].  
The data cannot be passed to the model in this form.  
To avoid bias toward the original winner/loser ordering, I shuffle the data so that the resulting columns are ['player_A'], ['player_B'], and ['player_A_won'].  

In [15]:
def transform_names(df: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """
    Transforms a tennis matches dataframe into the following format:
    jedna obserwacja na mecz: player_A vs player_B

    Additionally:
    - automatically maps `winner_*` and `loser_*` columns
    - randomly swaps `player_A` and `player_B` (data symmetrization)

    Parameters:
    df (pd.DataFrame): dataframe z kolumnami winner_* i loser_*
    random_state (int): seed for reproducibility

    Returns:
    pd.DataFrame
    """

    np.random.seed(random_state)

    result_df = df.copy()

    # --- podstawowe mapowanie ---
    result_df["player_A"] = result_df["winner_name"]
    result_df["player_B"] = result_df["loser_name"]
    result_df["player_A_won"] = 1

    winner_cols = [col for col in df.columns if col.startswith("winner_")]
    loser_cols = [col for col in df.columns if col.startswith("loser_")]

    # mapowanie kolumn
    for w_col in winner_cols:
        base_name = w_col.replace("winner_", "")
        l_col = f"loser_{base_name}"

        if l_col in df.columns:
            result_df[f"player_A_{base_name}"] = result_df[w_col]
            result_df[f"player_B_{base_name}"] = result_df[l_col]

    # --- SYMETRYZACJA ---
    mask = np.random.rand(len(result_df)) < 0.5

    # zamiana kolumn player_A_* i player_B_*
    for col in result_df.columns:
        if col.startswith("player_A_"):
            base = col.replace("player_A_", "")
            col_B = f"player_B_{base}"

            if col_B in result_df.columns:
                temp = result_df.loc[mask, col].copy()
                result_df.loc[mask, col] = result_df.loc[mask, col_B]
                result_df.loc[mask, col_B] = temp

    # swap player labels
    temp_A = result_df.loc[mask, "player_A"].copy()
    result_df.loc[mask, "player_A"] = result_df.loc[mask, "player_B"]
    result_df.loc[mask, "player_B"] = temp_A

    # flip the target
    result_df.loc[mask, "player_A_won"] = 0

    # remove the original columns
    result_df = result_df.drop(columns=winner_cols + loser_cols, errors="ignore")

    return result_df

matches_features = transform_names(matches_features)

### ADD ELO RATING

Nie kazdy turniej ma tyle samo meczy.  
Przyklad:  
for Grand Slams, the draw includes 128 players, so the match order looks like this:  
R1 -> R2 -> R3 -> R4 -> QF -> SF -> F  

for smaller tournaments, however, the draw is typically 32 players, so the match order looks like this:  
R1 -> R2 -> QF -> SF -> F  

Therefore, I add a column that captures the order in which matches were played within a tournament.  

In [16]:
def add_round_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds an `index_round` column to the dataframe to number rounds
    within a given tournament and year.

    Logic:
    - extracts the year from the `tourney_date` column
    - maps round labels to a numeric order
    - sorts data by tournament name, year, and round order
    - creates a round index within each `(tourney_name, year)` group
    - removes helper columns

    Parameters:
    df (pd.DataFrame): dataframe containing columns
        `tourney_date`, `round`, `tourney_name`

    Returns:
    pd.DataFrame: dataframe with the added `index_round` column
    """
    required_columns = {"tourney_date", "round", "tourney_name"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"Brakuje wymaganych kolumn: {sorted(missing_columns)}")

    result_df = df.copy()

    # add year for grouping
    result_df["year"] = result_df["tourney_date"].astype(str).str[:4]

    # map round order
    round_order = {
        "QF": 1,
        "R128": 2,
        "R64": 3,
        "R32": 4,
        "R16": 5,
        "QF": 6,
        "SF": 7,
        "F": 8,
        "RR": 10,
        "BR": 10
    }

    # add a helper column with round order
    result_df["round_order"] = result_df["round"].map(round_order)

    # sort the data
    result_df = result_df.sort_values(["tourney_name", "year", "round_order"])

    # assign the round index within each tournament and year
    result_df["index_round"] = (
        result_df.groupby(["tourney_name", "year"])["round_order"]
        .rank(method="dense")
        .astype(int)
    )

    # remove helper columns
    result_df = result_df.drop(columns=["year", "round_order"])

    return result_df

matches_features = add_round_index(matches_features)

Now we can add Elo rating

In [17]:
def compute_elo(df):
    df = df.sort_values(by=['tourney_date', 'tourney_name', 'index_round']).reset_index(drop=True)

    elo = {}
    init_rating=1500.0
    k_factor=32

    elo_A_before = []
    elo_B_before = []
    elo_A_after  = []
    elo_B_after  = []

    for _, row in df.iterrows():
        pA = row['player_A']
        pB = row['player_B']

        # ratings before match
        RA = elo.get(pA, init_rating)
        RB = elo.get(pB, init_rating)

        elo_A_before.append(RA)
        elo_B_before.append(RB)

        # estimated winning A
        EA = 1.0 / (1.0 + 10 ** ((RB - RA) / 400.0))
        EB = 1.0 - EA

        # real score
        if row['player_A_won'] == 1:
            SA, SB = 1.0, 0.0
        elif row['player_A_won'] == 0:
            SA, SB = 0.0, 1.0

        # update elo
        RA_new = RA + k_factor * (SA - EA)
        RB_new = RB + k_factor * (SB - EB)

        elo[pA] = RA_new
        elo[pB] = RB_new

        elo_A_after.append(RA_new)
        elo_B_after.append(RB_new)

    df["elo_A_before"] = elo_A_before
    df["elo_B_before"] = elo_B_before
    df["elo_A_after"] = elo_A_after
    df["elo_B_after"] = elo_B_after

    return df

matches_features = compute_elo(matches_features)

### CALCULATE COLUMN DIFFERENCES

In [18]:
def calculate_diff_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates difference features (`player_A - player_B`) for selected columns
    and removes the original `player_A_*` and `player_B_*` columns.

    Parameters:
    df (pd.DataFrame)

    Returns:
    pd.DataFrame
    """

    result_df = df.copy()

    # lista bazowych cech
    base_features = [
        "ht",
        "age",
        "rank",
        "rank_points"
    ]

    # compute differences for standard columns
    for feature in base_features:
        col_A = f"player_A_{feature}"
        col_B = f"player_B_{feature}"

        if col_A in result_df.columns and col_B in result_df.columns:
            result_df[f"diff_{feature}"] = result_df[col_A] - result_df[col_B]

    # special columns (Elo)
    if "elo_A_before" in result_df.columns and "elo_B_before" in result_df.columns:
        result_df["diff_elo_before"] = (
            result_df["elo_A_before"] - result_df["elo_B_before"]
        )

    # remove the original columns player_A_* i player_B_*
    cols_to_drop = [
        col for col in result_df.columns
        if col != "player_A_won" and (col.startswith("player_A_") or col.startswith("player_B_"))
    ]

    # additionally remove Elo columns if they exist
    cols_to_drop += ["elo_A_before", "elo_B_before"]

    result_df = result_df.drop(columns=cols_to_drop, errors="ignore")

    return result_df

matches_features = calculate_diff_features(matches_features)

### FINAL DROP OF UNNECESSARY COLUMNS

In [19]:
matches_features.drop(columns=['tourney_id', 'tourney_name', 'player_A', 'player_B', 'index_round', 'elo_A_after', 'elo_B_after'], inplace=True)

### CHANGE CATEGORICAL COLUMNS TO NUMERIC COLUMNS

In [20]:
matches_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40682 entries, 0 to 40681
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   surface           40682 non-null  object 
 1   draw_size         40682 non-null  int64  
 2   tourney_level     40682 non-null  object 
 3   tourney_date      40682 non-null  int64  
 4   round             40682 non-null  object 
 5   player_A_won      40682 non-null  int64  
 6   diff_ht           40682 non-null  int64  
 7   diff_age          40682 non-null  float64
 8   diff_rank         40682 non-null  float64
 9   diff_rank_points  40682 non-null  float64
 10  diff_elo_before   40682 non-null  float64
dtypes: float64(4), int64(4), object(3)
memory usage: 3.4+ MB


In [21]:
def encode_categorical(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detects categorical columns (`dtype='object'`) and converts them
    na zmienne zero-jedynkowe (one-hot encoding).

    Parameters:
    df (pd.DataFrame)

    Returns:
    pd.DataFrame
    """

    result_df = df.copy()

    # find categorical columns
    cat_cols = [
        col for col in result_df.columns
        if result_df[col].dtype == "object"
    ]

    # one-hot encoding
    result_df = pd.get_dummies(result_df, columns=cat_cols)

    return result_df

matches_features = encode_categorical(matches_features)

In [22]:
matches_features.shape

(40682, 34)

### SAVE DATA

In [23]:
matches_features.to_csv('/Users/martapriv/Documents/PUM/inzynierka_code/Tennis_Scores_Predictions/data/04_feature/matches_features.csv')